##### 工具注册
> 注意：
>
> 在LangChain中，作为工具的函数有两个保留的参数名，你的自定义参数不能与之重复！他们是：
>
> • config：用来传递运行时配置
>
> •runtime：用来传递运行时上下文
>
具体可参考官方文档：保留参数名 （Beserved argument names），后续我们会讲到这两个参数的用法。

In [1]:
# 1.加载环境变量
from typing import Literal

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool
from pydantic import BaseModel, Field
from rich import print as rich_print

load_dotenv(override=True)

# 2.初始化模型
model = init_chat_model(model="deepseek-v4-flash", extra_body={"thinking": {"type": "disabled"}})


# 3.定义工具
# 描述方式一：tool装饰器属件夹描述
@tool("square_root", description="计算数字x平方根")
def square_root(x: float) -> float:
    return x**0.5


# 描述方式二：用文档注释来描达
@tool
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """获取当前天气和可选天气预报。

    Args:
        location：城市名称或坐标
        units：温度单位
        include_forecast：包括天气预报吗
    """
    temp = 22 if units == "celsius" else 72
    result = f"[1]Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


# 方式三：基于 BaseModel 描述工具参
# 例如一个查询天气的 tool
class WeatherInput(BaseModel):
    """查询天气的输入参数。"""

    location: str = Field(description="城市名称或坐标")
    units: Literal["celsius", "fahrenheit"] = Field(default="celsius", description="温度单位偏好")
    include_forecast: bool = Field(default=False, description="是否包括未来5日预报")


# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather2(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"[2]Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "InNext 5 days: Sunny"
    return result


tools = [square_root, get_weather, get_weather2]
# 4.创建Agent，绑定模型和工具
agent = create_agent(model=model, tools=tools)

# 5.invoke调用Agent，传入前Dict，里面有messages
# response = agent.invoke({"messages": [HumanMessage("帮我计算429的平方根")]})
response = agent.invoke({"messages": [HumanMessage("杭州接下来的天气如何？")]})
rich_print(response)

{
    'messages': [
        HumanMessage(
            content='杭州接下来的天气如何？',
            additional_kwargs={},
            response_metadata={},
            id='76dc9a8d-470c-438d-93ea-71e0f0fbe301'
        ),
        AIMessage(
            content='我来查一下杭州的天气预报。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 80,
                    'prompt_tokens': 500,
                    'total_tokens': 580,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 244
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                'id': '56cd66e3-6a77-4b97-9b5b-c70e98aec4d4',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e7f8-fe72-7426-b4d0-c2eb38898571-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'location': '杭州', 'units': 'celsius', 'include_forecast': True},
                    'id': 'call_00_sHDhh3rlaSfxYLW5b02w6868',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 500,
                'output_tokens': 80,
                'total_tokens': 580,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='[1]Current weather in 杭州: 22 degrees C\nNext 5 days: Sunny',
            name='get_weather',
            id='9397a166-f888-4a10-b01a-0bf9adafe64e',
            tool_call_id='call_00_sHDhh3rlaSfxYLW5b02w6868'
        ),
        AIMessage(
            content='杭州的天气情况如下：\n\n**当前天气：** 22°C\n\n**未来5天预报：** 晴天 
☀️\n\n整体来看，接下来几天杭州天气都不错，以晴天为主，气温舒适（当前22°C），适合出行或户外活动。不过预报信息比较简
略，没有给出每天的具体温度和昼夜温差，建议出门前再留意一下当天的详细预报，早晚可能需要注意增减衣物。\n\n需要我帮你
查其他城市，或者换算成华氏度吗？',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 104,
                    'prompt_tokens': 611,
                    'total_tokens': 715,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 384,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 227
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                'id': 'a4182dde-4a85-4d6a-81db-fe5b489cf803',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a1e7f9-0160-75b3-bf4d-e07c287e8166-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 611,
                'output_tokens': 104,
                'total_tokens': 715,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {}
            }
        )
    ]
}